# Tools


TOOLS IN LLM – SIMPLE SUMMARY
====================================

What is a Tool?
---------------
A Tool is an external function that the LLM can request to execute.
The model does NOT run code by itself.
It only decides WHEN a function should be called and returns a structured
JSON request to call it.

Flow:
User → LLM → (requests tool) → Our code runs function → Result returned → LLM continues response

How to Define a Tool
--------------------
You pass a "tools" parameter when calling the model.

Example:

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a flight ticket to a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city name"
                    }
                },
                "required": ["destination_city"]
            }
        }
    }
]

Important:
- name → must match your Python function name
- description → helps the model know when to use it
- parameters → defines the JSON structure the model must send

-----------------------------------------------------

How to Use a Tool
-----------------

1) Send the request with tools=tools

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

2) Check if the model wants to call a tool:

if response.choices[0].finish_reason == "tool_calls":

"tool_calls" means:
The model did NOT finish normally.
It wants us to execute a function.

3) Extract the tool call:

message = response.choices[0].message
tool_call = message.tool_calls[0]

4) Run the correct function:

if tool_call.function.name == "get_ticket_price":
    arguments = json.loads(tool_call.function.arguments)
    city = arguments["destination_city"]
    result = get_ticket_price(city)

5) Return the result back to the model:

messages.append(message)

messages.append({
    "role": "tool",
    "content": result,
    "tool_call_id": tool_call.id
})

6) Call the model again so it can continue the answer.

-----------------------------------------------------

Why use a while loop?
---------------------
Because the model can call multiple tools in a row.
So we use:

while finish_reason == "tool_calls":

-----------------------------------------------------

Key Things to Understand
------------------------
- The model does NOT execute code.
- We execute the function.
- We must return the result with role="tool".
- The model can call the same function multiple times.
- Tools turn a chatbot into a real application.

Without Tools → Just text responses.
With Tools → Real actions (database, API calls, bookings, updates, etc.).

This is the foundation of building Agents.
"""

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [5]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [6]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [7]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [8]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool
There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [9]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [10]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Paris
